# Modelo para predecir si se han abierto las ventanas en la siguiente hora

In [23]:
from tensorflow.keras.models import load_model

model_temperatura = load_model('model_predict_temperaturas.keras', compile=False)


OSError: SavedModel file does not exist at: model_predict_temperaturas.keras/{saved_model.pbtxt|saved_model.pb}

In [20]:
import pandas as pd
from datetime import time
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('entrenamiento_neuronal_calefaccion_gold.csv')

scaler = MinMaxScaler()

df["temperatura_predicha"] = model_temperatura.predict(scaler.fit_transform(df[['sensor.sensor_temperatura_1_humidity',
       'sensor.sensor_temperatura_1_pressure',
       'sensor.sensor_temperatura_1_temperature', 'sensor_puerta_1 Puerta',
       'suma_ventanas_arriba', 'suma_ventanas_abajo', 'azimuth_mean',
       'elevacion_sol', 'temperatura_exterior', 'porcentaje_nubes',
    'hora', 'part_of_day', 'mes', 'season',
       'hora_sin', 'hora_cos', 'dia_1', 'dia_2', 'dia_3', 'dia_4', 'dia_5',
       'dia_6', 'dia_1', 'dia_2', 'dia_3', 'dia_4', 'dia_5', 'dia_6']]))


# Si la suma de suma_ventanas_arriba y suma_ventanas_abajo y puerta de la siguiente fila es mayor a 15 minutos 
df["ventanas_abiertas_siguiente_hora"] = (
    (df["suma_ventanas_arriba"].shift(-1) + df["suma_ventanas_abajo"].shift(-1) + df["sensor_puerta_1 Puerta"].shift(-1)) > 1200
)

df['horas'] = pd.to_datetime(df['time']).dt.time
df['dia_semana'] = pd.to_datetime(df['time']).dt.weekday  # 0 = lunes, 6 = domingo


def calefaccion(row):
    if 7 <= row['hora'] < 11 and row['temperatura_calefaccion_y'] < 22:
        return True
    elif 11 <= row['hora'] < 16 and row['temperatura_calefaccion_y'] < 21:
        return True
    elif 16 <= row['hora'] < 18 and row['temperatura_calefaccion_y'] < 21.6:
        return True
    elif 18 <= row['hora'] < 20.67 and row['temperatura_calefaccion_y'] < 22:
        return True
    else:
        return False

# Aplica la función para crear la columna 'calefaccion_encendida'
df['calefaccion_encendida'] = df.apply(calefaccion, axis=1)

# Filtramos cuando la calefacción está encendida
df = df[df['calefaccion_encendida'] == True]

# Filtra entre las 7:00 y las 21:00 y solo días entre semana (lunes a viernes)
df = df[(df['horas'] >= time(7, 0)) & 
        (df['horas'] <= time(21, 0)) & 
        (df['dia_semana'] < 5)]  # 0-4 para lunes a viernes

# Hacemos drop de hora despues porque se usa en la prediccion de la temperatura
df.drop(columns=['hora'], inplace=True) 

# Muestra el resultado
df.drop(columns=['horas','dia_semana'], inplace=True)

X_df = df.drop(columns=["time","ventanas_abiertas_siguiente_hora", "temperatura_calefaccion_y", "media_humedad","media_presion","media_temperatura"], inplace=True)
y_df = df["ventanas_abiertas_siguiente_hora"]

X_df.to_csv('X_df.csv', index=False)

NameError: name 'model_temperatura' is not defined

In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import make_classification
import pandas as pd
import numpy as np
import itertools

# Dividir en train, val, test
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y_df, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Parámetros para probar
hidden_layers_list = [[32], [64, 32], [128, 64, 32]]
activations = ['relu', 'tanh']
optimizers_list = [
    ('adam', 0.001),
    ('sgd', 0.01),
    ('rmsprop', 0.0005),
]

results = []

# Generar combinaciones
combinations = list(itertools.product(hidden_layers_list, activations, optimizers_list))

for i, (hidden_layers_config, activation, (opt_name, lr)) in enumerate(combinations, 1):
    print(f"\n🔧 Probando modelo {i}/{len(combinations)}: capas={hidden_layers_config}, activation={activation}, optimizer={opt_name}, lr={lr}")

    model = models.Sequential()
    model.add(layers.Input(shape=(X_df.shape[1],)))
    for units in hidden_layers_config:
        model.add(layers.Dense(units, activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))

    # Elegir optimizador
    if opt_name == 'adam':
        opt = optimizers.Adam(learning_rate=lr)
    elif opt_name == 'sgd':
        opt = optimizers.SGD(learning_rate=lr)
    elif opt_name == 'rmsprop':
        opt = optimizers.RMSprop(learning_rate=lr)

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0, validation_data=(X_val, y_val))

    # Evaluación
    y_pred_probs = model.predict(X_test).ravel()
    y_pred = (y_pred_probs > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        'layers': str(hidden_layers_config),
        'activation': activation,
        'optimizer': opt_name,
        'lr': lr,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1
    })

# Mostrar resultados ordenados por F1
df_resultados = pd.DataFrame(results)
df_sorted = df_resultados.sort_values(by='f1_score', ascending=False).reset_index(drop=True)
print("\nTop resultados por F1 score:")
print(df_sorted.head(10).to_string(index=False))


NameError: name 'X_df' is not defined

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import make_classification
import pandas as pd
import numpy as np



# Dividir en train, val, test
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y_df, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# ======== Solo UNA configuración ========
hidden_layers_config = [64, 32]
activation = 'relu'
opt_name = 'adam'
lr = 0.001

print(f"\n🔧 Entrenando red con: capas={hidden_layers_config}, activation={activation}, optimizer={opt_name}, lr={lr}")

model = models.Sequential()
model.add(layers.Input(shape=(X_df.shape[1],)))
for units in hidden_layers_config:
    model.add(layers.Dense(units, activation=activation))
model.add(layers.Dense(1, activation='sigmoid'))

# Elegir optimizador
if opt_name == 'adam':
    opt = optimizers.Adam(learning_rate=lr)
elif opt_name == 'sgd':
    opt = optimizers.SGD(learning_rate=lr)
elif opt_name == 'rmsprop':
    opt = optimizers.RMSprop(learning_rate=lr)

model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

# Entrenamiento
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0, validation_data=(X_val, y_val))
